# W9 Exercise 1 - Reinforcement Learning Basic Concepts

This practice notebook follows the Week 9 Reinforcement Learning lecture. You will implement small pieces of the agent-environment loop, returns, value functions, Bellman evaluation, temporal-difference learning, and exploration.

Use only the Python standard library. Complete each TODO before running the check cell below it.



## Learning outcomes

After finishing this exercise, you should be able to:

- represent states, actions, rewards, and transitions;
- simulate an episode under a policy;
- compute discounted returns;
- estimate state values from experience;
- apply a Bellman policy-evaluation backup;
- apply a TD(0) update;
- use epsilon-greedy exploration.



## Environment: Treasure Corridor

The agent moves in a one-dimensional corridor with five states:

$
S = \{0, 1, 2, 3, 4\}
$

- State $0$ is a bad terminal state with reward $-1$ when entered.
- State $4$ is a good terminal state with reward $+1$ when entered.
- States $1, 2, 3$ are non-terminal.
- Every non-terminal move has a small step cost $-0.02$.
- The action set is $A = \{\text{left}, \text{right}\}$.
- The chosen action succeeds with probability $0.80$; with probability $0.20$, the agent moves in the opposite direction.



In [1]:
from collections import Counter, defaultdict
import random

# Hệ số chiết khấu: quyết định mức độ phần thưởng tương lai ảnh hưởng đến tổng lợi ích.
GAMMA = 0.90

# Tất cả các trạng thái có thể có trong hành lang.
STATES = (0, 1, 2, 3, 4)

# Các trạng thái kết thúc sẽ làm dừng một episode.
TERMINALS = {0, 4}

# Agent có thể di chuyển sang trái hoặc sang phải.
ACTIONS = ("left", "right")


def is_terminal(state):
    """Trả về True nếu state là trạng thái kết thúc episode."""
    # Chỉ cần kiểm tra state có nằm trong TERMINALS hay không để biết đó là trạng thái kết thúc.
    return state in TERMINALS


print("States:", STATES)
print("Terminal states:", TERMINALS)
print("Actions:", ACTIONS)



States: (0, 1, 2, 3, 4)
Terminal states: {0, 4}
Actions: ('left', 'right')


## Exercise 1 - Transition model

Implement $P(s' \mid s, a)$ and $R(s, a, s')$ in the function transition_distribution.

The function must return a list of tuples:

$
(p, s', r)
$

For example, from state $s = 2$ with action $a = \text{right}$, the agent should usually move to $s' = 3$, but sometimes move to $s' = 1$.



In [4]:
def transition_distribution(state, action):
    """Trả về các kết quả ngẫu nhiên có thể xảy ra cho một cặp cặp trạng thái-hành động."""
    # Trạng thái kết thúc không di chuyển nữa và không sinh thêm phần thưởng.
    if state in TERMINALS:
        return [(1.0, state, 0.0)]

    # TODO 1: chuyển action thành hướng di chuyển dự định.
    # Gợi ý: left là -1, right là +1.
    intended_move = -1 if action == "left" else 1

    # TODO 2: xác định hướng di chuyển ngược lại.
    opposite_move = -intended_move

    # Danh sách này lưu tất cả kết quả có thể xảy ra của action ngẫu nhiên.
    outcomes = []

    # Di chuyển đúng hướng xảy ra với xác suất 0.80; di chuyển ngược hướng xảy ra với xác suất 0.20.
    for probability, move in [(0.80, intended_move), (0.20, opposite_move)]:
        # TODO 3: tính next_state và giới hạn nó trong biên của hành lang.
        next_state = max(0, min(4, state + move))

        # TODO 4: gán reward khi agent đi vào next_state.
        # Đi vào 0 nhận -1.0; đi vào 4 nhận +1.0; các trạng thái khác nhận -0.02.
        if next_state == 0:
            reward = -1.0  # Đi vào trạng thái xấu
        elif next_state == 4:
            reward = 1.0   # Đi vào trạng thái tốt
        else:
            reward = -0.02 # Chi phí nhỏ cho mỗi bước đi trong trạng thái thường

        # Lưu một kết quả có thể xảy ra dưới dạng (xác suất, next_state, reward).
        outcomes.append((probability, next_state, reward))

    # Trả về toàn bộ phân phối xác suất trên các kết quả.
    return outcomes

In [5]:
# Kiểm tra Bài 1 sau khi hoàn thành các TODO.
# Lấy thử một phân phối chuyển trạng thái đã biết.
dist = transition_distribution(2, "right")

# Một action ở trạng thái chưa kết thúc nên có hai kết quả ngẫu nhiên.
assert len(dist) == 2

# Tổng các xác suất phải bằng 1.
assert abs(sum(prob for prob, _, _ in dist) - 1.0) < 1e-12

# Từ state 2, action right thường đến 3 và đôi khi đến 1.
assert dist[0][1] == 3 and dist[1][1] == 1

# Đi vào trạng thái kết thúc xấu sẽ nhận reward âm.
assert transition_distribution(1, "left")[0][2] == -1.0

# Đi vào trạng thái kết thúc tốt sẽ nhận reward dương.
assert transition_distribution(3, "right")[0][2] == 1.0

print("Exercise 1 checks passed.")

Exercise 1 checks passed.


## Exercise 2 - Sampling a transition and running an episode

The helper function step samples one transition from your distribution $P(s' \mid s, a)$. Then you will implement two policies and use them to generate episodes.

A deterministic policy is a mapping:

$
\pi(s) = a
$



In [6]:
def step(state, action, rng):
    """Lấy mẫu một next_state và reward từ phân phối transition."""
    # Sinh một số ngẫu nhiên trong khoảng [0, 1).
    sample = rng.random()

    # Cộng dồn xác suất cho đến khi chạm khoảng chứa mẫu ngẫu nhiên.
    total = 0.0

    # Duyệt qua tất cả kết quả ngẫu nhiên có thể xảy ra.
    for probability, next_state, reward in transition_distribution(state, action):
        # Cộng thêm khối xác suất của kết quả hiện tại.
        total += probability

        # Nếu mẫu rơi vào khoảng này thì chọn kết quả hiện tại.
        if sample <= total:
            return next_state, reward

    # Dòng dự phòng cho các sai số rất nhỏ do số thực dấu phẩy động.
    probability, next_state, reward = transition_distribution(state, action)[-1]
    return next_state, reward


def policy_go_right(state):
    """Policy tất định luôn di chuyển về phía trạng thái kết thúc tốt."""
    # TODO: trả về action luôn đi về phía trạng thái kết thúc tốt.
    return "right"


def policy_random(state, rng):
    """Policy ngẫu nhiên chọn đều giữa các action."""
    # TODO: trả về một action ngẫu nhiên từ ACTIONS.
    return rng.choice(ACTIONS)

def run_episode(policy, start_state=2, seed=0, max_steps=20):
    """Sinh một episode bằng cách đi theo policy từ start_state."""
    # Dùng bộ sinh số ngẫu nhiên cục bộ để kết quả có thể tái lập.
    rng = random.Random(seed)

    # Episode bắt đầu từ start_state đã cho.
    state = start_state

    # Lưu transition dưới dạng (state, action, reward, next_state).
    episode = []

    # Dừng sau max_steps để tránh vòng lặp vô hạn khi policy chưa tốt.
    for _ in range(max_steps):
        # Nếu gặp trạng thái kết thúc thì episode dừng ngay.
        if is_terminal(state):
            break

        # Một số policy cần rng; policy tất định thì không cần.
        try:
            action = policy(state, rng)
        except TypeError:
            action = policy(state)

        # Lấy mẫu phản hồi của môi trường đối với action đã chọn.
        next_state, reward = step(state, action, rng)

        # Ghi lại transition để dùng cho học hoặc phân tích sau này.
        episode.append((state, action, reward, next_state))

        # Cập nhật state hiện tại sang next_state.
        state = next_state

    # Trả về toàn bộ quỹ đạo của episode.
    return episode


def print_episode(episode):
    """In trace transition ở dạng dễ đọc."""
    # Đánh số các transition để nhìn rõ từng bước thời gian.
    for t, (state, action, reward, next_state) in enumerate(episode):
        print(f"t={t:02d}: s={state}, a={action:>5}, r={reward:+.2f}, s_next={next_state}")

In [7]:
# Kiểm tra Bài 2 sau khi hoàn thành các TODO.
# Policy tất định này phải luôn chọn right.
assert policy_go_right(2) == "right"

# Policy ngẫu nhiên phải trả về một action hợp lệ.
rng = random.Random(1)
assert policy_random(2, rng) in ACTIONS

# Chạy một episode phải tạo ra ít nhất một transition.
sample_episode = run_episode(policy_go_right, seed=5)
assert len(sample_episode) > 0

# In episode mẫu để quan sát trực quan.
print_episode(sample_episode)
print("Exercise 2 checks passed.")

t=00: s=2, a=right, r=-0.02, s_next=3
t=01: s=3, a=right, r=+1.00, s_next=4
Exercise 2 checks passed.


## Exercise 3 - Discounted return

For rewards $r_1, r_2, r_3, \ldots$, the discounted return is:

$
G_0 = r_1 + \gamma r_2 + \gamma^2 r_3 + \cdots
$

Equivalently, for a finite episode:

$
G_0 = \sum_{t=0}^{T-1} \gamma^t r_{t+1}
$



In [9]:
def discounted_return(rewards, gamma=GAMMA):
    """Tính discounted return cho một chuỗi reward."""
    # Cộng dồn tổng reward đã được nhân trọng số chiết khấu.
    total = 0.0

    # Reward đầu tiên có trọng số gamma^0 = 1.
    discount = 1.0

    # Xử lý các reward theo thứ tự thời gian.
    for reward in rewards:
        # TODO: cộng reward đã chiết khấu vào total.
        total += discount * reward
        discount *= gamma

    # Trả về tổng reward chiết khấu cuối cùng.
    return total


# Thử hàm này trên một episode.
episode = run_episode(policy_go_right, seed=7)

# Chỉ lấy cột reward từ episode.
rewards = [reward for _, _, reward, _ in episode]

print("Rewards:", rewards)
print("Discounted return:", discounted_return(rewards))


Rewards: [-0.02, 1.0]
Discounted return: 0.88


In [10]:
# Kiểm tra Bài 3 sau khi hoàn thành TODO.
# Với gamma=0.5, tổng kỳ vọng là 1 + 0.5 + 0.25 = 1.75.
assert abs(discounted_return([1.0, 1.0, 1.0], gamma=0.5) - 1.75) < 1e-12

# Reward âm chỉ có một bước sẽ không bị thay đổi bởi chiết khấu.
assert abs(discounted_return([-1.0], gamma=0.9) + 1.0) < 1e-12

print("Exercise 3 checks passed.")

Exercise 3 checks passed.


## Exercise 4 - Monte Carlo value estimation

A value function estimates the expected return from each state under a policy:

$
V_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]
$

In this exercise, use complete episodes to estimate state values by averaging observed returns-to-go.



In [16]:
def returns_to_go(episode, gamma=GAMMA):
    """Trả về danh sách các cặp (state, return_from_that_state)."""
    # Biến result lưu một mẫu huấn luyện cho mỗi state đã đi qua.
    result = []

    # Return tích lũy được tính ngược từ cuối episode.
    running_return = 0.0

    # Duyệt ngược để mỗi reward được kết hợp với return tương lai.
    for state, action, reward, next_state in reversed(episode):
        # TODO: cập nhật running_return bằng reward và gamma.
        running_return = reward + gamma * running_return
        # Lưu mẫu (state, return_from_that_state) vào result.
        result.append((state, running_return))

    # Đảo ngược lại về thứ tự thời gian trước khi trả về.
    return list(reversed(result))


def monte_carlo_values(policy, episodes=200, seed=12):
    """Ước lượng V_pi(s) bằng cách lấy trung bình các return-to-go đã lấy mẫu."""
    # Dùng bộ sinh số ngẫu nhiên có thể tái lập cho các trạng thái bắt đầu.
    rng = random.Random(seed)

    # Lưu tất cả mẫu return quan sát được cho từng state.
    returns_by_state = defaultdict(list)

    # Sinh nhiều episode để giảm nhiễu do lấy mẫu.
    for episode_index in range(episodes):
        # Chỉ bắt đầu từ các state chưa kết thúc.
        start_state = rng.choice([1, 2, 3])

        # Dùng seed thay đổi để mỗi episode khác nhau nhưng vẫn tái lập được.
        episode = run_episode(policy, start_state=start_state, seed=seed + episode_index)

        # Chuyển episode thành các mẫu có giám sát: state -> return-to-go.
        for state, value_sample in returns_to_go(episode):
            returns_by_state[state].append(value_sample)

    # Lấy trung bình các mẫu đã thu thập cho từng state.
    return {
        state: sum(samples) / len(samples)
        for state, samples in returns_by_state.items()
        if samples
    }

# Ước lượng value từ một số ít episode để xem nhanh kết quả.
mc_values = monte_carlo_values(policy_go_right, episodes=50)

# In các value đã làm tròn theo thứ tự state.
print(dict(sorted((state, round(value, 3)) for state, value in mc_values.items())))

{1: 0.391, 2: 0.705, 3: 0.939}


In [17]:
# Kiểm tra Bài 4 sau khi hoàn thành các TODO.
# Episode ngắn này có return-to-go đã biết.
test_episode = [(2, "right", -0.02, 3), (3, "right", 1.0, 4)]

# Với gamma=0.9, return từ state 2 là -0.02 + 0.9 * 1.0 = 0.88.
rtg = returns_to_go(test_episode, gamma=0.9)
assert len(rtg) == 2
assert rtg[0][0] == 2 and abs(rtg[0][1] - 0.88) < 1e-12
assert rtg[1][0] == 3 and abs(rtg[1][1] - 1.0) < 1e-12

print("Exercise 4 checks passed.")

Exercise 4 checks passed.


## Exercise 5 - Bellman policy evaluation

For a fixed policy $\pi$, Bellman evaluation repeatedly applies:

$
V_\pi(s) = \sum_{s'} P(s' \mid s, \pi(s)) \left[ R(s, \pi(s), s') + \gamma V_\pi(s') \right]
$



In [19]:
def bellman_backup_for_policy(state, policy, values, gamma=GAMMA):
    """Tính một Bellman backup cho policy cố định."""
    # Trong bài này, trạng thái kết thúc không có giá trị tương lai.
    if is_terminal(state):
        return 0.0

    # Policy quyết định action nào sẽ được đánh giá tại state này.
    action = policy(state)

    # Cộng dồn kỳ vọng trên tất cả next_state có thể xảy ra.
    expected_value = 0.0

    # TODO: dùng transition_distribution để tính kỳ vọng Bellman.
    for probability, next_state, reward in transition_distribution(state, action):
        expected_value += probability * (reward + gamma * values[next_state])
        
    # Trả về phần thưởng tức thời kỳ vọng cộng với giá trị tương lai đã chiết khấu.
    return expected_value


def evaluate_policy_with_bellman(policy, iterations=50):
    """Ước lượng V_pi bằng cách lặp Bellman backup."""
    # Khởi tạo value của tất cả state bằng 0.
    values = {state: 0.0 for state in STATES}

    # Các lượt quét lặp lại sẽ lan truyền reward terminal ngược qua không gian trạng thái.
    for _ in range(iterations):
        # Dùng bản sao để mọi state được cập nhật từ value của vòng lặp trước.
        new_values = values.copy()

        # Áp dụng một Bellman backup cho từng state.
        for state in STATES:
            new_values[state] = bellman_backup_for_policy(state, policy, values)

        # Thay các ước lượng cũ bằng ước lượng mới.
        values = new_values

    # Trả về các ước lượng value cuối cùng.
    return values


# Đánh giá policy always-go-right bằng mô hình đã biết.
bellman_values = evaluate_policy_with_bellman(policy_go_right)

# In các value đã làm tròn để dễ đọc.
print({state: round(value, 3) for state, value in bellman_values.items()})



{0: 0.0, 1: 0.284, 2: 0.694, 3: 0.921, 4: 0.0}


In [20]:
# Kiểm tra Bài 5 sau khi hoàn thành TODO.
# Khi giá trị tương lai bằng 0, state 3 với action right có kỳ vọng value 0.8*(+1) + 0.2*(-0.02) = 0.796.
values_zero = {state: 0.0 for state in STATES}
assert abs(bellman_backup_for_policy(3, policy_go_right, values_zero) - 0.796) < 1e-12

# Sau nhiều lần backup, state 3 nên có value dương.
assert evaluate_policy_with_bellman(policy_go_right)[3] > 0.0

print("Exercise 5 checks passed.")

Exercise 5 checks passed.


## Exercise 6 - Temporal-difference update

TD(0) updates the value of the current state after observing one transition:

$
V(s) \leftarrow V(s) + \alpha \left[r + \gamma V(s') - V(s)\right]
$

The bracketed term is the TD error:

$
\delta = r + \gamma V(s') - V(s)
$



In [21]:
def td_update(current_value, reward, next_value, alpha=0.10, gamma=GAMMA):
    """Trả về ước lượng value mới sau một bước TD(0)."""
    # TODO: tính một bước cập nhật TD(0) và trả về current value mới.
    return current_value + alpha * (reward + gamma * next_value - current_value)


def td_learning(policy, episodes=300, alpha=0.10, seed=21):
    """Ước lượng state value từ các transition đã lấy mẫu bằng TD(0)."""
    # Dùng bộ sinh số ngẫu nhiên có thể tái lập cho trạng thái bắt đầu và transition.
    rng = random.Random(seed)

    # Các state value chưa biết được khởi tạo bằng 0.
    values = defaultdict(float)

    # Lặp nhiều episode để tinh chỉnh ước lượng.
    for episode_index in range(episodes):
        # Bắt đầu từ một non-trạng thái kết thúc ngẫu nhiên.
        state = rng.choice([1, 2, 3])

        # Giới hạn độ dài episode để an toàn.
        for _ in range(20):
            # Dừng khi môi trường đạt trạng thái kết thúc.
            if is_terminal(state):
                break

            # Đi theo policy cố định.
            action = policy(state)

            # Quan sát một transition từ môi trường.
            next_state, reward = step(state, action, rng)

            # Áp dụng cập nhật TD cho ước lượng của state hiện tại.
            values[state] = td_update(values[state], reward, values[next_state], alpha=alpha)

            # Chuyển sang next_state.
            state = next_state

    # Trả về bảng value đã học được.
    return values


# Chạy TD learning sau khi cài đặt td_update.
td_values = td_learning(policy_go_right)

# In các value đã làm tròn để kiểm tra.
print({state: round(td_values[state], 3) for state in STATES})



{0: 0.0, 1: -0.105, 2: 0.564, 3: 0.882, 4: 0.0}


In [22]:
# Kiểm tra Bài 6 sau khi hoàn thành TODO.
# Bắt đầu từ 0 với reward 1 và alpha 0.1 thì value nên tăng lên 0.1.
assert abs(td_update(0.0, 1.0, 0.0, alpha=0.1, gamma=0.9) - 0.1) < 1e-12

# Dòng này kiểm tra toàn bộ biểu thức: 2 + 0.5 * (-1 + 0.5*4 - 2) = 1.5.
assert abs(td_update(2.0, -1.0, 4.0, alpha=0.5, gamma=0.5) - 1.5) < 1e-12

print("Exercise 6 checks passed.")

Exercise 6 checks passed.


## Exercise 7 - Exploration with epsilon-greedy action selection

An agent that always exploits current estimates may stop exploring too early. Implement epsilon-greedy action selection for action utilities $Q(s, a)$:

$
a = \begin{cases}
\text{random action}, & \text{with probability } \epsilon \\
\arg\max_a Q(s, a), & \text{with probability } 1 - \epsilon
\end{cases}
$



In [23]:
def epsilon_greedy_action(q_values, state, epsilon, rng):
    """Chọn action bằng chiến lược epsilon-tham lam."""
    # TODO: với xác suất epsilon, chọn một action ngẫu nhiên.
    # Ngược lại, chọn action có Q-value lớn nhất.
    if rng.random() < epsilon:
        return rng.choice(ACTIONS)
    
    return max(ACTIONS, key=lambda action: q_values[(state, action)])

# Tạo một Q-table nhỏ cho một state.
q_values = defaultdict(float)

# Action left hiện được ước lượng là kém.
q_values[(2, "left")] = -0.20

# Action right hiện được ước lượng là tốt hơn.
q_values[(2, "right")] = 0.50

# Dùng bộ sinh số ngẫu nhiên có thể tái lập cho thí nghiệm.
rng = random.Random(99)

# Đếm số lần mỗi action được chọn.
counts = Counter(epsilon_greedy_action(q_values, 2, epsilon=0.25, rng=rng) for _ in range(1000))

print(counts)

Counter({'right': 847, 'left': 153})


In [24]:
# Kiểm tra Bài 7 sau khi hoàn thành TODO.
# Với epsilon=0, agent luôn khai thác action tốt nhất.
rng = random.Random(0)
assert epsilon_greedy_action(q_values, 2, epsilon=0.0, rng=rng) == "right"

# Với epsilon=1, lấy mẫu lặp lại cuối cùng nên xuất hiện mọi action.
rng = random.Random(0)
actions_seen = {epsilon_greedy_action(q_values, 2, epsilon=1.0, rng=rng) for _ in range(50)}
assert actions_seen == set(ACTIONS)

print("Exercise 7 checks passed.")


Exercise 7 checks passed.


## Reflection questions

Answer these in your own words:

1. Why does $V_\pi(3)$ tend to be higher than $V_\pi(1)$ under policy_go_right?

    Under the policy_go_right, the agent always moves towards the right, which leads to the terminal state 4 that provides a positive reward. State 3 is closer to state 4 than state 1, so it has a higher expected return due to being more likely to reach the positive reward sooner. In contrast, state 1 is further away from the positive reward and has a higher chance of receiving negative rewards from moving left towards state 0.

2. What information is used by Bellman evaluation that Monte Carlo averaging does not use directly?

    Bellman evaluation uses the immediate reward and the discounted value of the next state to update the value of the current state. Monte Carlo averaging, on the other hand, estimates the value of a state by averaging the returns (total discounted rewards) from all episodes that visit that state.

3. Why can epsilon-greedy exploration help an action-utility learner?

    Epsilon-greedy exploration allows the agent to occasionally choose a random action instead of the action with the highest estimated value. This helps the learner to explore the environment and discover potentially better actions that it might not have tried if it always exploited the current best-known action. It prevents the agent from getting stuck in a local optimum and encourages learning about the true value of all actions.

4. What happens when $\gamma$ is close to $0$? What happens when $\gamma$ is close to $1$?

    When $\gamma$ is close to $0$, the agent becomes myopic and focuses almost entirely on immediate rewards, ignoring future rewards. This can lead to suboptimal behavior if the best long-term strategy involves sacrificing short-term rewards. When $\gamma$ is close to $1$, the agent values future rewards more heavily, which encourages it to consider long-term consequences of its actions. However, if $\gamma$ is exactly $1$, it may lead to convergence issues in certain environments, especially those with infinite horizons or where rewards can accumulate indefinitely.